In [1]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

sns.set_style("whitegrid")
plt.rcParams.update({
    "figure.dpi": 100, "savefig.dpi": 300, "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 11,
})

DISTRICT_ORDER = ["Burera", "Gakenke", "Musanze", "Ngororero",
                   "Nyabihu", "Rubavu", "Rutsiro"]
SEASON_ORDER = ["Season A (Rainy)", "Season B (Rainy)", "Dry Season"]
SEASON_COLORS = {"Season A (Rainy)": "#2E86AB",
                  "Season B (Rainy)": "#4CAF91",
                  "Dry Season": "#D9A441"}

df = pd.read_csv("Processed Data.csv", parse_dates=["date"])
df = df.sort_values(["Grid_id", "Year", "Month"]).reset_index(drop=True)

# Recreate the stable spatial grid-cell key (needed for persistence analysis)
df["grid_cell_id"] = df["Latitude"].astype(str) + "_" + df["Longitude"].astype(str)

print(f"Loaded {df.shape[0]} rows. Total extreme events: "
      f"{df['is_extreme_rainfall'].sum()} "
      f"({df['is_extreme_rainfall'].mean()*100:.2f}% of all records)")

# ==========================================================================
# (a) TREND ANALYSIS: Mann-Kendall Test + Sen's Slope Annual Rainfall
# ==========================================================================
print("\n" + "=" * 140)
print("(a) TREND ANALYSIS: ARE EXTREME RAINFALL EVENTS INCREASING FREQUENTLY?")
print("=" * 140)

yearly_counts = df.groupby("Year")["is_extreme_rainfall"].sum().reset_index()
yearly_counts.columns = ["Year", "Extreme Event Count"]
yearly_counts.to_csv("Yearly Extreme Event Trend.csv", index=False)
print(yearly_counts.to_string(index=False))


def mann_kendall_test(x):
    x = np.asarray(x, dtype=float)
    n = len(x)
    s = 0
    for k in range(n - 1):
        s += np.sum(np.sign(x[k + 1:] - x[k]))
    unique_vals, counts = np.unique(x, return_counts=True)
    tie_term = np.sum(counts * (counts - 1) * (2 * counts + 5))
    var_s = (n * (n - 1) * (2 * n + 5) - tie_term) / 18
    if s > 0:
        z = (s - 1) / np.sqrt(var_s)
    elif s < 0:
        z = (s + 1) / np.sqrt(var_s)
    else:
        z = 0
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return s, z, p


def sens_slope(years, values):
    years = np.asarray(years, dtype=float)
    values = np.asarray(values, dtype=float)
    n = len(years)
    slopes = [
        (values[j] - values[i]) / (years[j] - years[i])
        for i in range(n - 1) for j in range(i + 1, n)
    ]
    return np.median(slopes)


mk_s, mk_z, mk_p = mann_kendall_test(yearly_counts["Extreme Event Count"].values)
sen_slope = sens_slope(yearly_counts["Year"].values,
                        yearly_counts["Extreme Event Count"].values)

alpha = 0.05
mk_result = pd.DataFrame([{
    "Test": "Mann-Kendall Trend Test",
    "S statistic": mk_s,
    "Z statistic": round(mk_z, 3),
    "p-value": round(mk_p, 4),
    "Sen's Slope (events/year)": round(sen_slope, 3),
    "Trend Direction": "Increasing" if mk_s > 0 else ("Decreasing" if mk_s < 0 else "None"),
    "Statistical Significance (α=0.05)": "Yes" if mk_p < alpha else "No",
}])
print("\n" + mk_result.to_string(index=False))
mk_result.to_csv("Mann Kendall Trend Test.csv", index=False)

# Figure: Annual trend with Sen's slope line
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.bar(yearly_counts["Year"], yearly_counts["Extreme Event Count"],
       color="#D64545", alpha=0.75, label="Extreme Events/Year")
trend_line = sen_slope * (yearly_counts["Year"] - yearly_counts["Year"].iloc[0]) + \
             yearly_counts["Extreme Event Count"].iloc[0]
ax.plot(yearly_counts["Year"], trend_line, color="#1A1A1A", linewidth=2,
        linestyle="--",
        label=f"Sen's Slope trend ({sen_slope:+.2f} events/yr, "
              f"p={mk_p:.3f})")
ax.set_title("Frequency of Extreme Rainfall Events (1983–2021)")
ax.set_xlabel("Year")
ax.set_ylabel("Number of Extreme Events")
ax.legend()
plt.tight_layout()
plt.savefig("Extreme Event Trend.png")
plt.close()
print("\nSaved: Extreme Event Trend.png")

# ==========================================================================
# (b) ASSOCIATION: Chi-square tests of Association (Season, District vs Extreme event)
# ==========================================================================
print("\n" + "=" * 140)
print("(b) ASSOCIATION: CHI-SQUARE TESTS OF INDEPENDENCE")
print("=" * 140)

# Season x Extreme
season_crosstab = pd.crosstab(df["Season"], df["is_extreme_rainfall"])
chi2_s, p_s, dof_s, _ = stats.chi2_contingency(season_crosstab)
chi_season_result = pd.DataFrame([{
    "Test": "Season vs Extreme Rainfall",
    "Chi-square statistic": round(chi2_s, 2),
    "Degrees of Freedom": dof_s,
    "p-value": "<0.0001" if p_s < 0.0001 else round(p_s, 4),
    "Statistical Significance (α=0.05)": "Yes" if p_s < alpha else "No",
    "Interpretation": "Season and extreme rainfall event occurrence are NOT independent"
                       if p_s < alpha else "Significant association NOT detected",
}])
print(chi_season_result.to_string(index=False))
chi_season_result.to_csv("Chi Square Season Association.csv", index=False)

# District x Extreme
district_crosstab = pd.crosstab(df["District"], df["is_extreme_rainfall"])
chi2_d, p_d, dof_d, _ = stats.chi2_contingency(district_crosstab)
chi_district_result = pd.DataFrame([{
    "Test": "District vs Extreme Rainfall",
    "Chi-square statistic": round(chi2_d, 2),
    "Degrees of Freedom": dof_d,
    "p-value": "<0.0001" if p_d < 0.0001 else round(p_d, 4),
    "Statistical Significance (α=0.05)": "Yes" if p_d < alpha else "No",
    "Interpretation": "District and Extreme Rainfall Event Occurrence are NOT independent"
                       if p_d < alpha else "Significant Association NOT detected",
}])
print("\n" + chi_district_result.to_string(index=False))
chi_district_result.to_csv("Chi Square District Association.csv", index=False)

# ==========================================================================
# (c) MAGNITUDE: how far above threshold do extreme events go?
# ==========================================================================
print("\n" + "=" * 140)
print("(c) MAGNITUDE: EXCESSIVE RAINFALL BEYOND THE 95TH-PERCENTILE THRESHOLD")
print("=" * 140)

extreme_df = df[df["is_extreme_rainfall"] == 1].copy()
extreme_df["excess_mm"] = extreme_df["pr"] - extreme_df["district_pr_p95"]
extreme_df["excess_pct"] = (extreme_df["excess_mm"] / extreme_df["district_pr_p95"]) * 100

overall_mag = {
    "Group": "Overall",
    "N Events": len(extreme_df),
    "Mean Excess (mm)": round(extreme_df["excess_mm"].mean(), 1),
    "Median Excess (mm)": round(extreme_df["excess_mm"].median(), 1),
    "Max Excess (mm)": round(extreme_df["excess_mm"].max(), 1),
    "Mean Excess (%)": round(extreme_df["excess_pct"].mean(), 1),
}
season_mag = []
for s_ in SEASON_ORDER:
    sub = extreme_df[extreme_df["Season"] == s_]
    if len(sub) > 0:
        season_mag.append({
            "Group": s_, "N Events": len(sub),
            "Mean Excess (mm)": round(sub["excess_mm"].mean(), 1),
            "Median Excess (mm)": round(sub["excess_mm"].median(), 1),
            "Max Excess (mm)": round(sub["excess_mm"].max(), 1),
            "Mean Excess (%)": round(sub["excess_pct"].mean(), 1),
        })
district_mag = []
for d in DISTRICT_ORDER:
    sub = extreme_df[extreme_df["District"] == d]
    if len(sub) > 0:
        district_mag.append({
            "Group": d, "N Events": len(sub),
            "Mean Excess (mm)": round(sub["excess_mm"].mean(), 1),
            "Median Excess (mm)": round(sub["excess_mm"].median(), 1),
            "Max Excess (mm)": round(sub["excess_mm"].max(), 1),
            "Mean Excess (%)": round(sub["excess_pct"].mean(), 1),
        })

magnitude_summary = pd.DataFrame([overall_mag] + season_mag + district_mag)
print(magnitude_summary.to_string(index=False))
magnitude_summary.to_csv("Event Magnitude Summary.csv", index=False)

# Figure: magnitude distribution (histogram)
fig, ax = plt.subplots(figsize=(9.5, 5.5))
sns.histplot(extreme_df["excess_mm"], bins=30, kde=True, color="#D64545", ax=ax)
ax.axvline(extreme_df["excess_mm"].median(), color="#1A1A1A", linestyle="--",
           label=f"Median excess = {extreme_df['excess_mm'].median():.1f} mm")
ax.set_title("Distribution of Excessive Rainfall Beyond District 95th Percentile\n"
              "(Extreme Events Only)")
ax.set_xlabel("Excess Rainfall (mm beyond threshold)")
ax.set_ylabel("Frequency")
ax.legend()
plt.tight_layout()
plt.savefig("Event Magnitude Distribution.png")
plt.close()
print("\nSaved: Event Magnitude Distribution.png")

# Figure: magnitude by season and district (boxplots side by side)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
sns.boxplot(data=extreme_df, x="Season", y="excess_mm", order=SEASON_ORDER,
            palette=[SEASON_COLORS[s] for s in SEASON_ORDER], ax=axes[0])
axes[0].set_title("Excessive Rainfall by Season")
axes[0].set_xlabel("Season")
axes[0].set_ylabel("Excessive Rainfall (mm)")

sns.boxplot(data=extreme_df, x="District", y="excess_mm", order=DISTRICT_ORDER,
            palette="Blues", ax=axes[1])
axes[1].set_title("Excessive Rainfall by District")
axes[1].set_xlabel("District")
axes[1].set_ylabel("Excessive Rainfall (mm)")
axes[1].tick_params(axis="x", rotation=30)
for lbl in axes[1].get_xticklabels():
    lbl.set_ha("right")
fig.suptitle("Event Magnitude (Excessive Rainfall) by Season and District",
             fontweight="normal", fontsize=14)
plt.tight_layout()
plt.savefig("Event Magnitude by Season & District.png")
plt.close()
print("Saved: Event Magnitude by Season & District.png")

# ==========================================================================
# (d) PERSISTENCE: do extreme events cluster into multi-month runs?
# ==========================================================================
print("\n" + "=" * 140)
print("(d) PERSISTENCE: EVENT DURATION / CLUSTERING ANALYSIS")
print("=" * 140)

run_lengths = []
for cell_id, group in df.groupby("grid_cell_id"):
    flags = group.sort_values(["Year", "Month"])["is_extreme_rainfall"].values
    run = 0
    for f in flags:
        if f == 1:
            run += 1
        else:
            if run > 0:
                run_lengths.append(run)
            run = 0
    if run > 0:
        run_lengths.append(run)

run_lengths = np.array(run_lengths)
persistence_summary = pd.DataFrame([{
    "Total Discrete Events (runs)": len(run_lengths),
    "Single-Month Events (n)": int((run_lengths == 1).sum()),
    "Single-Month Events (%)": round((run_lengths == 1).mean() * 100, 1),
    "Multi-Month Events (n)": int((run_lengths > 1).sum()),
    "Multi-Month Events (%)": round((run_lengths > 1).mean() * 100, 1),
    "Mean Run Length (months)": round(run_lengths.mean(), 2),
    "Max Run Length (months)": int(run_lengths.max()),
}])
print(persistence_summary.to_string(index=False))
persistence_summary.to_csv("Clustering Analysis Summary.csv", index=False)

# Figure: run length distribution
fig, ax = plt.subplots(figsize=(8.5, 5.5))
run_counts = pd.Series(run_lengths).value_counts().sort_index()
ax.bar(run_counts.index, run_counts.values, color="#D64545")
ax.bar_label(ax.containers[0])
ax.set_title("Distribution of Extreme Event Run Lengths\n"
              "(Consecutive Rainfall Excess Months at the Same Grid Cell)")
ax.set_xlabel("Run Length (consecutive months)")
ax.set_ylabel("Number of Events")
ax.set_xticks(run_counts.index)
plt.tight_layout()
plt.savefig("Clustering Analysis.png")
plt.close()
print("\nSaved: Clustering Analysis.png")

# ==========================================================================
# Plus FIGURE: Year x Month calendar heatmap of extreme event counts
# ==========================================================================
pivot = df.pivot_table(index="Year", columns="Month",
                        values="is_extreme_rainfall", aggfunc="sum")
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                 "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
fig, ax = plt.subplots(figsize=(10, 11))
sns.heatmap(pivot, cmap="Blues", linewidths=0.4, linecolor="white",
            cbar_kws={"label": "Extreme Rainfall Event Count"}, ax=ax)
ax.set_xticklabels(month_labels)
ax.set_title("Calendar Heatmap of Extreme Rainfall Events\n(Year × Month)")
ax.set_xlabel("Month")
ax.set_ylabel("Year")
plt.tight_layout()
plt.savefig("Extreme Event Calendar Heatmap.png")
plt.close()
print("Saved: Extreme Event Calendar Heatmap.png")

print("\n" + "=" * 140)
print("RAINFALL EVENT IDENTIFICATION & CHARACTERISATION Successfully Complete")
print("=" * 140)


Loaded 18720 rows. Total extreme events: 927 (4.95% of all records)

(a) TREND ANALYSIS: ARE EXTREME RAINFALL EVENTS INCREASING FREQUENTLY?
 Year  Extreme Event Count
 1983                   19
 1984                    5
 1985                   30
 1986                   16
 1987                   20
 1988                   40
 1989                   16
 1990                   27
 1991                   12
 1992                   16
 1993                   12
 1994                    6
 1995                   17
 1996                   20
 1997                   48
 1998                   12
 1999                   22
 2000                   18
 2001                   21
 2002                   31
 2003                   45
 2004                   34
 2005                   12
 2006                   37
 2007                   21
 2008                    9
 2009                   13
 2010                   28
 2011                   28
 2012                   35
 2013                  